#### Imports & Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# Lock randomness so weights initialize exactly the same way every time
np.random.seed(42)

#### Data Ingestion & State Filtering

In [24]:
# Load exact local Excel files
df_1 = pd.read_excel(r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\MBP ControllerData 0521760 Overlock.xlsx")
df_2 = pd.read_excel(r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\Synthetic Overlock Breakdowns.xlsx")

# Combine them into one giant raw dataset first
master_df = pd.concat([df_1, df_2], ignore_index=True)

# Handle the blank spaces (Any row without a breakdown becomes "Healthy")
master_df['Breakdown'] = master_df['Breakdown'].fillna('Healthy')

# Keep only the 15 allowed states
allowed_states = [
    "Healthy", "Needle Breakages", "High Foot Pressure", "Cut/Needle Hole", 
    "Thread Breakages", "Pneumatic Issues", "Thread Jamming", 
    "Code Uneven", "Roping", "Oil Mark", "Skip Stitches/Slip", 
    "Gathering/Puckering", "Waveness", "Binding/Seam Open", "Blade Broken"
]
master_df = master_df[master_df['Breakdown'].isin(allowed_states)].copy()

# Separate into pure datasets (zero mixing)
healthy_df = master_df[master_df['Breakdown'] == 'Healthy'].copy()
breakdown_df = master_df[master_df['Breakdown'] != 'Healthy'].copy()

# Shuffling the dataframes to ensure randomness before splitting
breakdown_df = breakdown_df.sample(frac=1, random_state=42).reset_index(drop=True)
healthy_df = healthy_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 7. Split 80/20 on each dataset separately to maintain class balance in both Train and Test sets
split_h = int(len(healthy_df) * 0.8)
split_b = int(len(breakdown_df) * 0.8)

train_healthy = healthy_df.iloc[:split_h]
test_healthy  = healthy_df.iloc[split_h:]

train_breakdown = breakdown_df.iloc[:split_b]
test_breakdown  = breakdown_df.iloc[split_b:]

# 8. Combine into final Train and Test DataFrames
train_df = pd.concat([train_healthy, train_breakdown], ignore_index=True)
test_df = pd.concat([test_healthy, test_breakdown], ignore_index=True)

print(f"Train Set: {len(train_healthy)} Healthy + {len(train_breakdown)} Breakdowns = {len(train_df)} total rows")
print(f"Test Set: {len(test_healthy)} Healthy + {len(test_breakdown)} Breakdowns = {len(test_df)} total rows")

Train Set: 3962 Healthy + 562 Breakdowns = 4524 total rows
Test Set: 991 Healthy + 141 Breakdowns = 1132 total rows
